In [8]:
# ==========================================
# WEEK 1: DATA UNDERSTANDING & CLEANING
# ==========================================

import pandas as pd
import numpy as np

# 1. Load raw dataset
df = pd.read_csv('data/raw/Food delivery Dataset.csv')
initial_rows = len(df)

# 2. Data Inspection
print(f"Dataset Shape: {df.shape}")
print("\n--- Column Data Types & Non-Null Counts ---")
print(df.info())

missing_df = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df)) * 100
})
print("\n--- Missing Values Summary ---")
print(missing_df[missing_df['Missing_Count'] > 0])

# 3. Handle Missing Values
df['Delivery_person_Age'] = df['Delivery_person_Age'].fillna(df['Delivery_person_Age'].median())
df['Delivery_person_Ratings'] = df['Delivery_person_Ratings'].fillna(df['Delivery_person_Ratings'].median())
df['multiple_deliveries'] = df['multiple_deliveries'].fillna(df['multiple_deliveries'].mode()[0])

df['Weather_conditions'] = df['Weather_conditions'].fillna('Unknown')
df['Road_traffic_density'] = df['Road_traffic_density'].fillna('Unknown')
df['City'] = df['City'].fillna('Unknown')
df['Festival'] = df['Festival'].fillna('No')

# 4. Clean Coordinate Anomalies
valid_lat = (df['Restaurant_latitude'].between(6, 37)) & (df['Delivery_location_latitude'].between(6, 37))
valid_lon = (df['Restaurant_longitude'].between(68, 97)) & (df['Delivery_location_longitude'].between(68, 97))
df = df[valid_lat & valid_lon].copy()

# 5. Calculate Haversine Distance (in kilometers)
def calculate_haversine(lat1, lon1, lat2, lon2):
    R = 6371.0 # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

df['Distance_km'] = calculate_haversine(
    df['Restaurant_latitude'], df['Restaurant_longitude'],
    df['Delivery_location_latitude'], df['Delivery_location_longitude']
)

# 6. Export Cleaned Dataset
df.to_csv('data/processed/cleaned_food_delivery.csv', index=False)
print(f"\nCleaning complete! Initial rows: {initial_rows}, Cleaned rows: {len(df)}")

Dataset Shape: (45584, 20)

--- Column Data Types & Non-Null Counts ---
<class 'pandas.DataFrame'>
RangeIndex: 45584 entries, 0 to 45583
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID                           45584 non-null  str    
 1   Delivery_person_ID           45584 non-null  str    
 2   Delivery_person_Age          43730 non-null  float64
 3   Delivery_person_Ratings      43676 non-null  float64
 4   Restaurant_latitude          45584 non-null  float64
 5   Restaurant_longitude         45584 non-null  float64
 6   Delivery_location_latitude   45584 non-null  float64
 7   Delivery_location_longitude  45584 non-null  float64
 8   Order_Date                   45584 non-null  str    
 9   Time_Orderd                  43853 non-null  str    
 10  Time_Order_picked            45584 non-null  str    
 11  Weather_conditions           44968 non-null  str    
 12  Road_traffic_